In [1]:
print("Here begins the hybrid models notebook.")

Here begins the hybrid models notebook.


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.feature_selection import SelectFromModel

from xgboost import XGBRegressor

pd.set_option("display.max_columns", None)
df_ml = pd.read_csv("../../data/processed/world_happiness_ml.csv")

df_ml.head()

,Country,Year,Happiness_Score,GDP_per_Capita,Social_Support,Healthy_Life_Expectancy,Freedom,Generosity,Corruption_Perception,Unemployment_Rate,Education_Index,Population,Urbanization_Rate,Life_Satisfaction,Public_Trust,Mental_Health_Index,Income_Inequality,Public_Health_Expenditure,Climate_Index,Work_Life_Balance,Internet_Access,Crime_Rate,Political_Stability,Employment_Rate,Year_bin
0,China,2022,4.39,44984.68,0.53,71.11,0.41,-0.05,0.83,14.98,0.52,1311940760,78.71,8.88,0.34,76.44,46.06,8.92,62.75,8.59,74.40,70.30,0.29,61.38,2020
1,UK,2015,5.49,30814.59,0.93,63.14,0.89,0.04,0.84,19.46,0.83,1194240877,50.87,5.03,0.72,53.38,46.43,4.43,53.11,8.76,91.74,73.32,0.76,80.18,2015
2,Brazil,2009,4.65,39214.84,0.03,62.36,0.01,0.16,0.59,16.68,0.95,731100898,48.75,5.22,0.23,82.40,31.03,3.78,33.30,6.06,71.80,28.99,0.94,72.65,2005
3,France,2019,5.20,30655.75,0.77,78.94,0.98,0.25,0.63,2.64,0.70,1293957314,81.78,5.69,0.68,46.87,57.65,4.43,90.59,6.36,86.16,45.76,0.48,55.14,2015
4,China,2022,7.28,30016.87,0.05,50.33,0.62,0.18,0.92,7.70,0.92,1432971455,82.39,6.33,0.50,60.38,28.54,7.66,59.33,3.00,71.10,65.67,0.12,51.55,2020


In [2]:
# Inspect data structure
print(df_ml.shape)
print(df_ml.dtypes)

(4000, 25)
Country                          str
Year                           int64
Happiness_Score              float64
GDP_per_Capita               float64
Social_Support               float64
Healthy_Life_Expectancy      float64
Freedom                      float64
Generosity                   float64
Corruption_Perception        float64
Unemployment_Rate            float64
Education_Index              float64
Population                     int64
Urbanization_Rate            float64
Life_Satisfaction            float64
Public_Trust                 float64
Mental_Health_Index          float64
Income_Inequality            float64
Public_Health_Expenditure    float64
Climate_Index                float64
Work_Life_Balance            float64
Internet_Access              float64
Crime_Rate                   float64
Political_Stability          float64
Employment_Rate              float64
Year_bin                       int64
dtype: object


In [3]:
# Define predictors and target
# - Life_Satisfaction is the regression target
# - Country is excluded because it is a high-cardinality identifier
# - Year_bin is excluded because it duplicates the information in Year
X = df_ml.drop(columns=["Life_Satisfaction", "Country", "Year_bin"])
y = df_ml["Life_Satisfaction"]

# Check shapes
print(X.shape, y.shape)

# Check missing values in predictors and target
print("Missing values in X:", X.isna().sum().sum())
print("Missing values in y:", y.isna().sum())

(4000, 22) (4000,)
Missing values in X: 0
Missing values in y: 0


In [4]:
# Train/test split
# Since this is a regression task, stratification is not used
# The split is still useful for final model comparison, even though cross-validation
# is especially important for this small dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (3200, 22)
Test shape: (800, 22)


In [5]:
# Optional: cross-validation baseline for Linear Regression
# With only around 200 rows, cross-validation gives a more stable estimate
# than relying only on a single train/test split

cv = KFold(n_splits=5, shuffle=True, random_state=42)

linreg_cv = LinearRegression()

cv_results_lin = cross_validate(
    linreg_cv,
    X, y,
    cv=cv,
    scoring={
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2"
    }
)

print("Linear Regression CV results")
print("Mean MAE:", -cv_results_lin["test_mae"].mean())
print("Mean RMSE:", -cv_results_lin["test_rmse"].mean())
print("Mean R2:", cv_results_lin["test_r2"].mean())

Linear Regression CV results
Mean MAE: 1.2401107895846064
Mean RMSE: 1.4363265534361156
Mean R2: -0.0038631863228493213


In [6]:
# Hybrid 1: Random Forest feature selection
# Random Forest is used here as a nonlinear feature selector
# The final model will still be Linear Regression for interpretability

rf_selector = RandomForestRegressor(
    n_estimators=150,
    max_depth=6,
    min_samples_split=10,
    min_samples_leaf=4,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

rf_selector.fit(X_train, y_train)

selector = SelectFromModel(
    rf_selector,
    threshold="median"
)

selector.fit(X_train, y_train)

X_train_sel = selector.transform(X_train)
X_test_sel = selector.transform(X_test)

selected_features = X_train.columns[selector.get_support()]

print("Number of selected features:", len(selected_features))
print("\nSelected features:")
print(selected_features.tolist())

Number of selected features: 11

Selected features:
['Happiness_Score', 'GDP_per_Capita', 'Generosity', 'Population', 'Urbanization_Rate', 'Mental_Health_Index', 'Income_Inequality', 'Climate_Index', 'Work_Life_Balance', 'Crime_Rate', 'Political_Stability']


In [7]:
# Hybrid 1: Linear Regression on RF-selected features
# This combines nonlinear feature selection with a simple, interpretable regression model

linreg_hybrid_rf = LinearRegression()

linreg_hybrid_rf.fit(X_train_sel, y_train)
y_pred_hybrid1 = linreg_hybrid_rf.predict(X_test_sel)

print("Hybrid 1: RF Feature Selection -> Linear Regression")
print("MAE:", mean_absolute_error(y_test, y_pred_hybrid1))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_hybrid1)))
print("R2:", r2_score(y_test, y_pred_hybrid1))

Hybrid 1: RF Feature Selection -> Linear Regression
MAE: 1.2107234477544488
RMSE: 1.4109191305014896
R2: 0.003907958973309089


In [8]:
# Coefficients for Hybrid 1
# These coefficients help interpret which selected predictors are most strongly
# associated with life satisfaction in the final linear model

coef_df_rf = pd.DataFrame({
    "Feature": selected_features,
    "Coefficient": linreg_hybrid_rf.coef_
})

coef_df_rf["Abs_Coefficient"] = coef_df_rf["Coefficient"].abs()
coef_df_rf.sort_values("Abs_Coefficient", ascending=False).head(15)

,Feature,Coefficient,Abs_Coefficient
2,Generosity,2.867385e-01,2.867385e-01
10,Political_Stability,-6.326814e-02,6.326814e-02
0,Happiness_Score,2.900421e-02,2.900421e-02
6,Income_Inequality,-5.396405e-03,5.396405e-03
8,Work_Life_Balance,-4.428267e-03,4.428267e-03
5,Mental_Health_Index,2.516471e-03,2.516471e-03
9,Crime_Rate,-1.549742e-03,1.549742e-03
4,Urbanization_Rate,-8.532245e-04,8.532245e-04
7,Climate_Index,5.118201e-04,5.118201e-04
1,GDP_per_Capita,-3.574206e-10,3.574206e-10


In [9]:
# Hybrid 2: XGBoost feature selection
# XGBoost is used as an alternative nonlinear selector
# This allows comparison with Random Forest-based feature selection

xgb_selector = XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

xgb_selector.fit(X_train, y_train)

selector_xgb = SelectFromModel(
    xgb_selector,
    threshold="median"
)

selector_xgb.fit(X_train, y_train)

X_train_sel_xgb = selector_xgb.transform(X_train)
X_test_sel_xgb = selector_xgb.transform(X_test)

selected_features_xgb = X_train.columns[selector_xgb.get_support()]

print("Number of selected features:", len(selected_features_xgb))
print("\nSelected features:")
print(selected_features_xgb.tolist())

Number of selected features: 11

Selected features:
['GDP_per_Capita', 'Education_Index', 'Population', 'Urbanization_Rate', 'Public_Trust', 'Mental_Health_Index', 'Income_Inequality', 'Climate_Index', 'Crime_Rate', 'Political_Stability', 'Employment_Rate']


In [10]:
# Hybrid 2: Linear Regression on XGBoost-selected features
# This tests whether XGBoost-based feature selection produces better final
# linear predictions than Random Forest-based feature selection

linreg_hybrid_xgb = LinearRegression()

linreg_hybrid_xgb.fit(X_train_sel_xgb, y_train)
y_pred_hybrid2 = linreg_hybrid_xgb.predict(X_test_sel_xgb)

print("Hybrid 2: XGB Feature Selection -> Linear Regression")
print("MAE:", mean_absolute_error(y_test, y_pred_hybrid2))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_hybrid2)))
print("R2:", r2_score(y_test, y_pred_hybrid2))

Hybrid 2: XGB Feature Selection -> Linear Regression
MAE: 1.2128845573963525
RMSE: 1.4157481894611617
R2: -0.0029222254715999263


In [11]:
# Coefficients for Hybrid 2
# Again, coefficients are inspected after feature selection to interpret
# which variables remain important in the final linear model

coef_df_xgb = pd.DataFrame({
    "Feature": selected_features_xgb,
    "Coefficient": linreg_hybrid_xgb.coef_
})

coef_df_xgb["Abs_Coefficient"] = coef_df_xgb["Coefficient"].abs()
coef_df_xgb.sort_values("Abs_Coefficient", ascending=False).head(15)

,Feature,Coefficient,Abs_Coefficient
1,Education_Index,3.935139e-01,3.935139e-01
9,Political_Stability,-6.965061e-02,6.965061e-02
6,Income_Inequality,-5.416315e-03,5.416315e-03
5,Mental_Health_Index,2.477306e-03,2.477306e-03
10,Employment_Rate,-2.077464e-03,2.077464e-03
8,Crime_Rate,-1.494686e-03,1.494686e-03
4,Public_Trust,-9.478387e-04,9.478387e-04
3,Urbanization_Rate,-8.644392e-04,8.644392e-04
7,Climate_Index,5.621824e-04,5.621824e-04
0,GDP_per_Capita,-1.348658e-07,1.348658e-07


In [12]:
# Hybrid 3: Residual hybrid model
# fit a Linear Regression model, compute residuals on the training set, fit XGBoost on those residuals
# Final prediction = linear prediction + predicted residual correction

linreg_base = LinearRegression()
linreg_base.fit(X_train, y_train)

# Residuals from the linear model on training data
train_pred_lin = linreg_base.predict(X_train)
train_residuals = y_train - train_pred_lin

# XGBoost model on residuals
xgb_residual = XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

xgb_residual.fit(X_train, train_residuals)

# Final predictions on test set
y_pred_lin_test = linreg_base.predict(X_test)
y_pred_residual_test = xgb_residual.predict(X_test)
y_pred_hybrid3 = y_pred_lin_test + y_pred_residual_test

print("Hybrid 3: Linear Regression + Residual XGBoost")
print("MAE:", mean_absolute_error(y_test, y_pred_hybrid3))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_hybrid3)))
print("R2:", r2_score(y_test, y_pred_hybrid3))

Hybrid 3: Linear Regression + Residual XGBoost
MAE: 1.223743429936897
RMSE: 1.4266878834285521
R2: -0.01848156310461402


In [13]:
# Inspect baseline linear coefficients from the residual hybrid
# These remain useful for interpretation because the hybrid starts with a
# transparent linear structure before applying nonlinear residual correction

coef_df_residual_base = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": linreg_base.coef_
})

coef_df_residual_base["Abs_Coefficient"] = coef_df_residual_base["Coefficient"].abs()
coef_df_residual_base.sort_values("Abs_Coefficient", ascending=False).head(15)

,Feature,Coefficient,Abs_Coefficient
9,Education_Index,0.373387,0.373387
6,Generosity,0.281653,0.281653
3,Social_Support,-0.071622,0.071622
20,Political_Stability,-0.065485,0.065485
5,Freedom,0.043130,0.043130
7,Corruption_Perception,0.030953,0.030953
1,Happiness_Score,0.029718,0.029718
15,Public_Health_Expenditure,-0.009412,0.009412
14,Income_Inequality,-0.005287,0.005287
0,Year,0.004103,0.004103


In [14]:
# Hybrid 4: Stacking Regressor
# This combines the predictions of:
# - Linear Regression
# - Random Forest Regressor
# - XGBoost Regressor
# The final estimator is again Linear Regression

base_estimators = [
    ("lr", LinearRegression()),
    ("rf", RandomForestRegressor(
        n_estimators=100,
        max_depth=6,
        min_samples_split=10,
        min_samples_leaf=4,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    )),
    ("xgb", XGBRegressor(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    ))
]

stack_model = StackingRegressor(
    estimators=base_estimators,
    final_estimator=LinearRegression(),
    cv=5,
    n_jobs=-1
)

stack_model.fit(X_train, y_train)
y_pred_stack = stack_model.predict(X_test)

print("Hybrid 4: Stacking Regressor")
print("MAE:", mean_absolute_error(y_test, y_pred_stack))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_stack)))
print("R2:", r2_score(y_test, y_pred_stack))

Hybrid 4: Stacking Regressor
MAE: 1.2123759182841534
RMSE: 1.413422134557783
R2: 0.00037064222270877245


In [15]:
# Compare all hybrid models on the test set
# - Lower MAE and RMSE are better
# - Higher R2 is better

results = pd.DataFrame([
    {
        "Model": "RF selection -> Linear Regression",
        "MAE": mean_absolute_error(y_test, y_pred_hybrid1),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_hybrid1)),
        "R2": r2_score(y_test, y_pred_hybrid1),
    },
    {
        "Model": "XGB selection -> Linear Regression",
        "MAE": mean_absolute_error(y_test, y_pred_hybrid2),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_hybrid2)),
        "R2": r2_score(y_test, y_pred_hybrid2),
    },
    {
        "Model": "Linear Regression + Residual XGBoost",
        "MAE": mean_absolute_error(y_test, y_pred_hybrid3),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_hybrid3)),
        "R2": r2_score(y_test, y_pred_hybrid3),
    },
    {
        "Model": "Stacking Regressor",
        "MAE": mean_absolute_error(y_test, y_pred_stack),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_stack)),
        "R2": r2_score(y_test, y_pred_stack),
    }
])

results.sort_values("R2", ascending=False)

,Model,MAE,RMSE,R2
0,RF selection -> Linear Regression,1.210723,1.410919,0.003908
3,Stacking Regressor,1.212376,1.413422,0.000371
1,XGB selection -> Linear Regression,1.212885,1.415748,-0.002922
2,Linear Regression + Residual XGBoost,1.223743,1.426688,-0.018482


In [16]:
# Compare overlap between RF- and XGB-selected features
# This helps assess whether both nonlinear selectors identify a similar
# set of relevant predictors

rf_set = set(selected_features)
xgb_set = set(selected_features_xgb)

common_features = sorted(rf_set.intersection(xgb_set))
rf_only = sorted(rf_set - xgb_set)
xgb_only = sorted(xgb_set - rf_set)

print("Number of common selected features:", len(common_features))
print("\nCommon features:")
print(common_features)

print("\nFeatures selected only by RF:")
print(rf_only)

print("\nFeatures selected only by XGB:")
print(xgb_only)

Number of common selected features: 8

Common features:
['Climate_Index', 'Crime_Rate', 'GDP_per_Capita', 'Income_Inequality', 'Mental_Health_Index', 'Political_Stability', 'Population', 'Urbanization_Rate']

Features selected only by RF:
['Generosity', 'Happiness_Score', 'Work_Life_Balance']

Features selected only by XGB:
['Education_Index', 'Employment_Rate', 'Public_Trust']


In [17]:
# Cross-validation for the main hybrid models
# Because the dataset is small, cross-validation provides a more robust estimate
# of performance than a single split alone

models_cv = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        max_depth=6,
        min_samples_split=10,
        min_samples_leaf=4,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": XGBRegressor(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    )
}

cv_summary = []

for name, model in models_cv.items():
    scores = cross_validate(
        model,
        X, y,
        cv=cv,
        scoring={
            "mae": "neg_mean_absolute_error",
            "rmse": "neg_root_mean_squared_error",
            "r2": "r2"
        }
    )

    cv_summary.append({
        "Model": name,
        "CV_MAE": -scores["test_mae"].mean(),
        "CV_RMSE": -scores["test_rmse"].mean(),
        "CV_R2": scores["test_r2"].mean()
    })

cv_results = pd.DataFrame(cv_summary)
cv_results.sort_values("CV_R2", ascending=False)

,Model,CV_MAE,CV_RMSE,CV_R2
0,Linear Regression,1.240111,1.436327,-0.003863
1,Random Forest,1.243611,1.436820,-0.004563
2,XGBoost,1.244079,1.440017,-0.009090
